---
#### Web search
---

In [1]:
from openai import OpenAI

In [2]:
client = OpenAI()

In [3]:
response = client.chat.completions.create(
    model             = "gpt-4o-search-preview",  # Required for web search
    web_search_options= {},                       # Enables search
    messages          = [
        {"role": "user", "content": "Any new FDA-approved cancer treatment this week?"}
    ]
)

print(response.choices[0].message.content)

As of December 6, 2025, the U.S. Food and Drug Administration (FDA) has not announced any new cancer treatment approvals within the past week. The most recent approvals occurred in November 2025, including:

- **Ziftomenib (Komzifti):** Approved on November 13, 2025, for the treatment of adults with relapsed or refractory acute myeloid leukemia (AML) harboring a susceptible nucleophosmin 1 (NPM1) mutation. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Ziftomenib?utm_source=openai))

- **Sevabertinib (Hyrnuo):** Approved on November 19, 2025, for the treatment of HER2-mutant non-small cell lung cancer. ([drugs.com](https://www.drugs.com/l/GMrdNmPpZTI1763lD892XGKFoQ/gHLjs7rDhOs94qAjlOUtbw/rdOU1V47638E3gvzOx9BfOWA?utm_source=openai))

For the most current information on FDA approvals, it's advisable to consult the FDA's official announcements or reputable medical news sources. 


**Example with custom options**

In [4]:
# Summarize recent research on CAR-T therapy in India for leukemia patients.
# Give me latest updates about heart disease treatment in India.

response = client.chat.completions.create(
    model="gpt-4o-search-preview",
    messages=[
        {"role": "user", "content": "Summarize recent research on CAR-T therapy in India for leukemia patients."}
    ],
    web_search_options={
        "search_context_size": "high"
    }
)

print(response.choices[0].message.content)

Recent advancements in CAR-T cell therapy in India have significantly improved treatment options for leukemia patients. Notably, the development and approval of indigenous therapies have enhanced accessibility and affordability.

**Development and Approval of Indigenous CAR-T Therapies**

In October 2023, ImmunoACT, an IIT Bombay-incubated company, received approval from the Central Drugs Standard Control Organization (CDSCO) for NexCAR19, India's first humanized CD19-targeted CAR-T cell therapy. This therapy is designed to treat relapsed or refractory B-cell lymphomas and leukemia. Clinical trials involving 60 patients demonstrated a 70% overall response rate, with a favorable safety profile. NexCAR19 is priced between ₹30 to ₹40 lakh per patient, significantly lower than similar treatments available internationally. ([timesofindia.indiatimes.com](https://timesofindia.indiatimes.com/business/india-business/iit-b-startup-immunoacts-affordable-blood-cancer-therapy-gets-regulatory-approv

**Purpose of user_location**

This field allows web-enhanced GPT models (like gpt-4o-search-preview) to personalize search results based on your geographical context — useful for:

- Region-specific laws, treatments, or policies
- Local events, services, and prices
- Language/regional trends

In [5]:
def build_web_search_options(
    context_size: str = "medium",
    city: str     = None,
    region: str   = None,
    country: str  = None,
    timezone: str = None
) -> dict:
    """
    Builds a valid web_search_options dictionary for OpenAI web search.
    All location fields are optional, but country is recommended.

    context_size: one of "low", "medium", "high"
    """

    user_location = {
        "type": "approximate",
        "approximate": {}
    }

    if city:
        user_location["approximate"]["city"] = city
    if region:
        user_location["approximate"]["region"] = region
    if country:
        user_location["approximate"]["country"] = country
    if timezone:
        user_location["approximate"]["timezone"] = timezone

    return {
        "search_context_size": context_size,
        "user_location": user_location
    }

In [7]:
import json

options = build_web_search_options(
    context_size = "high",
    city         = "Bangalore",          # free text strings
    region       = "Karnataka",          # free text strings
    country      = "IN",                 # two-letter ISO country code, like US
    timezone     = "Asia/Kolkata"        # is an IANA timezone like America/Chicago
)

print(json.dumps(options, indent=2))

{
  "search_context_size": "high",
  "user_location": {
    "type": "approximate",
    "approximate": {
      "city": "Bangalore",
      "region": "Karnataka",
      "country": "IN",
      "timezone": "Asia/Kolkata"
    }
  }
}


In [8]:
response = client.chat.completions.create(
    model="gpt-4o-search-preview",
    #model="o4-mini",
    messages=[
        {"role": "user", "content": "Any new breakthroughs in diabetes treatment"}
    ],
    web_search_options=options
)

print(response.choices[0].message.content)

Recent advancements in diabetes treatment have introduced promising therapies and technologies aimed at improving patient outcomes.

**1. Introduction of Ozempic in India**

Novo Nordisk is set to launch Ozempic, a once-weekly semaglutide injection, in India this month. Ozempic is approved for type 2 diabetes management and is also used off-label for weight loss. This introduction comes as India faces rising rates of diabetes and obesity. ([reuters.com](https://www.reuters.com/business/healthcare-pharmaceuticals/novo-nordisk-gears-up-december-ozempic-launch-india-sources-say-2025-12-03/?utm_source=openai))

**2. Approval of Teplizumab for Type 1 Diabetes**

The European Medicines Agency has recommended approval for Sanofi's Teplizumab (Teizeild), a groundbreaking drug that delays the onset of stage 3 insulin-dependent type 1 diabetes. This marks the first treatment aimed at slowing disease progression. ([reuters.com](https://www.reuters.com/business/healthcare-pharmaceuticals/sanofis-t

No impact of location !!!

---
#### new-style tool calling format
---

the OpenAI /v1/responses endpoint, where you directly specify tools like:

- web_search_preview
- code_interpreter
- retrieval
- function_calling (via tool_choice)

You explicitly list tools in the request, and OpenAI decides how to use them based on your input.

In [9]:
response = client.responses.create(
    model ="gpt-4o",
    input ="What are the best restaurants around yelahanka?",
    tools =[
        {
            "type": "web_search_preview",
            "user_location": {
                "type": "approximate",
                "country": "IN",
                "city": "Bangalore",
                "region": "India"
            }
        }
    ]
)

print(response.output_text)

Yelahanka, a vibrant suburb in North Bangalore, offers a diverse culinary scene catering to various tastes. Here are some notable restaurants you might consider:

**[Nysa Sky Bar](https://www.google.com/maps/search/Nysa+Sky+Bar%2C+Yelahanka%2C+Bangalore?utm_source=openai)**
_Yelahanka, Bangalore_
A rooftop bar known for its breathtaking views, live music, signature cocktails, and a fusion menu, making it ideal for brunches, date nights, and weekend gatherings.

**[The Druid Garden](https://www.google.com/maps/search/The+Druid+Garden%2C+Yelahanka%2C+Bangalore?utm_source=openai)**
_Yelahanka, Bangalore_
Famous for its craft brews, wood-fired pizzas, and modern European-style cuisine, offering a blend of quality and innovation.

**[AB's – Absolute Barbecues](https://www.google.com/maps/search/AB%27s+%E2%80%93+Absolute+Barbecues%2C+Yelahanka%2C+Bangalore?utm_source=openai)**
_Yelahanka, Bangalore_
A popular chain known for its wide variety of barbecued meats and seafood, offering a buffet-

> This forces the model to use the web_search_preview tool with the specified location.


**Example for a Healthcare Query**

In [10]:
response = client.responses.create(
    model="gpt-4o",
    input="What are the latest diabetes treatment options in India?",
    tools=[
        {
            "type": "web_search_preview",
            "user_location": {
                "type": "approximate",
                "country": "IN",
                "city": "Mumbai",
                "region": "Maharashtra",
                "timezone": "Asia/Kolkata"
            }
        }
    ]
)

print(response.output_text)


As of December 2025, India has witnessed significant advancements in diabetes treatment, introducing several innovative medications and technologies to enhance patient care.

**1. Introduction of GLP-1 Receptor Agonists:**

- **Ozempic (Semaglutide):** Novo Nordisk launched Ozempic in India in December 2025. This once-weekly injectable medication is approved for managing type 2 diabetes and has shown efficacy in weight management. ([reuters.com](https://www.reuters.com/business/healthcare-pharmaceuticals/novo-nordisk-gears-up-december-ozempic-launch-india-sources-say-2025-12-03/?utm_source=openai))

- **Mounjaro (Tirzepatide):** Eli Lilly introduced Mounjaro in March 2025. This drug targets both GIP and GLP-1 receptors, offering a novel approach to metabolic health. ([reuters.com](https://www.reuters.com/business/healthcare-pharmaceuticals/eli-lilly-launches-weight-loss-drug-mounjaro-india-after-drug-regulator-approval-2025-03-20/?utm_source=openai))

- **Wegovy (Semaglutide):** Novo N

#### comparison with Old Format
    
| Feature                       | `chat.completions.create(...)`      | `responses.create(...)` (✅ new-style)        |
| ----------------------------- | ----------------------------------- | -------------------------------------------- |
| Endpoint                      | `/v1/chat/completions`              | `/v1/responses`                              |
| Tools specified?              | ❌ Indirect via `web_search_options` | ✅ Explicit via `tools=[...]`                 |
| Location-sensitive Web Search | ❌ Weak, sometimes ignored           | ✅ Strong control via `user_location`         |
| Multi-tool behavior           | ❌ No                                | ✅ Supports multiple tools (e.g., code + web) |


---
#### search_context_size in web_search_preview

**What it controls:**

How much text (context) is retrieved from the web to help generate the LLM response.

---

| Setting              | Context Detail        | Cost Impact        | Quality               | Latency     | Use Case Example                   |
| -------------------- | --------------------- | ------------------ | --------------------- | ----------- | ---------------------------------- |
| `"low"`              | Snippets or headlines | Cheapest         | ❗ Might miss nuance   | 🚀 Fastest  | Real-time headlines, location tips |
| `"medium"` (default) | Balanced context      | Moderate        |  Good for general Qs |  Balanced | Health/tech news, product queries  |
| `"high"`             | Full paragraphs/pages | Possibly billed |  Best for deep Qs   |  Slowest  | Medical advances, legal analysis   |


In [11]:
tools = [{
    "type": "web_search_preview",
    "user_location": {
        "type": "approximate",
        "city": "Delhi",
        "region": "Delhi",
        "country": "IN"
    },
    "search_context_size": "low"
}]

In [12]:
response = client.responses.create(
    model="gpt-4o",
    input="Are there any new clinical trials on reversing diabetes in India?",
    tools=tools
)

print(response.output_text)

Yes, there are several recent clinical trials and initiatives in India focused on reversing diabetes:

1. **Diabetes Remission in India (DiRemI) Study**: This prospective, open-label, matched-group trial evaluates a one-year online intensive lifestyle intervention combining dietary modifications, physical activity, psychological support, and medical management. The study aims to assess the impact on weight loss and diabetes remission among adults with type 2 diabetes. ([pubmed.ncbi.nlm.nih.gov](https://pubmed.ncbi.nlm.nih.gov/38941311/?utm_source=openai))

2. **CURE-DM Trial**: Conducted by the Diabetes Foundation, India, this two-year randomized control trial investigates the effects of a low-calorie, high-protein, low-carbohydrate vegetarian diet combined with exercise on reversing type 2 diabetes. The study focuses on weight loss and reduction of ectopic fat in the liver and pancreas. ([ichgcp.net](https://ichgcp.net/clinical-trials-registry/NCT05925946?utm_source=openai))

3. **Red